In [23]:
import pandas as pd

In [24]:
PATH = '../datasets/ai_vs_human_text_2026.csv'
df = pd.read_csv(PATH)

In [25]:
df.shape

(2000, 9)

In [26]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score, roc_curve)

In [27]:
df['label_enc'] = (df['label'] == 'ai').astype(int)
df.head()

,text_id,label,source_model,domain,text_content,topic_hint,word_count,avg_sentence_length,generation_method,label_enc
0,TXT_0001,human,human,social,can we talk about gene editing ethics for a se...,gene editing ethics,27,13.5,template+human_variation,0
1,TXT_0002,human,human,social,update on election integrity concerns: it's co...,election integrity concerns,20,10.0,template+human_variation,0
2,TXT_0003,ai,gemini-2.0,news,Analysts are closely watching developments rel...,climate change adaptation strategies,39,13.0,style_simulation,1
3,TXT_0004,human,human,academic,This paper examines genomic research breakthro...,genomic research breakthroughs,49,16.3,template+human_variation,0
4,TXT_0005,ai,gpt-4o,academic,Existing literature on student debt crisis has...,student debt crisis,42,10.8,style_simulation,1


In [28]:
X = df['text_content']
y = df['label_enc']

In [29]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [30]:
print(f"Train size : {len(X_train)}")
print(f"Test size : {len(X_test)}")
print(f"Train label balance : {y_train.value_counts().to_dict()}")
print(f"Test label balance : {y_test.value_counts().to_dict()}")

Train size : 1600
Test size : 400
Train label balance : {0: 1067, 1: 533}
Test label balance : {0: 267, 1: 133}


In [31]:
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(1, 2),
        max_features=10000,
        sublinear_tf=True,
        stop_words='english'
    )),
    ('clf', LogisticRegression(
        class_weight='balanced',
        max_iter=1000,
        random_state=42
    ))
])

pipeline.fit(X_train, y_train)
y_pred  = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]

print("=== Classification Report ===")
print(classification_report(y_test, y_pred, target_names=['Human', 'AI']))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_proba):.4f}")

=== Classification Report ===
              precision    recall  f1-score   support

       Human       1.00      1.00      1.00       267
          AI       1.00      1.00      1.00       133

    accuracy                           1.00       400
   macro avg       1.00      1.00      1.00       400
weighted avg       1.00      1.00      1.00       400

ROC-AUC Score: 1.0000


In [32]:
import joblib

joblib.dump(pipeline, '../app/ml/trained/ai_detection_model.pkl')

['../app/ml/trained/ai_detection_model.pkl']